In [1]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [2]:
train_data = pd.read_csv("samsum-train.csv")
val_data = pd.read_csv("samsum-validation.csv")

train_data.head()

,id,dialogue,summary
0,13818513,Amanda: I baked cookies. Do you want some?\r\...,Amanda baked cookies and will bring Jerry some...
1,13728867,Olivia: Who are you voting for in this electio...,Olivia and Olivier are voting for liberals in ...
2,13681000,"Tim: Hi, what's up?\r\nKim: Bad mood tbh, I wa...",Kim may try the pomodoro technique recommended...
3,13730747,"Edward: Rachel, I think I'm in ove with Bella....",Edward thinks he is in love with Bella. Rachel...
4,13728094,Sam: hey overheard rick say something\r\nSam:...,"Sam is confused, because he overheard Rick com..."


In [3]:
train_data.shape

(14732, 3)

In [4]:
val_data.shape

(818, 3)

In [5]:
# have to preprocess the data, like remove the unwanted elements like HTML tags, new line elements.

## Random Sampling

In [6]:
train_data = train_data.sample(n = 4000, random_state = 42).reset_index(drop = True)
val_data = val_data.sample(n = 500, random_state = 42).reset_index(drop = True)

In [7]:
train_data.shape

(4000, 3)

# Data Preprocessing

In [8]:
import re

def clean_data(text):
    text = re.sub(r"\r\n"," ", text)  # line replacement
    text = re.sub(r"\s+"," ", text)  # empty space replacement
    text = re.sub(r"<.*?>"," ", text)  # HTML tag replacement
    text = text.strip().lower()

    return text

In [9]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)


val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
val_data["summary"] = val_data["summary"].apply(clean_data)

In [10]:
train_data["dialogue"][0]

"violet: hi! i came across this austin's article and i thought that you might find it interesting violet:   claire: hi! :) thanks, but i've already read it. :) claire: but thanks for thinking about me :)"

# Tokenization

In [11]:
# tokenization means converting our text into numbers
# like for "I ate an apple" --> 32,405,111,727

In [12]:
# We will be using T5Tokenizer for T5 model and it has ~32k tokens in its vocabulary.

In [13]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

In [14]:
# function for raw data -> tokens for fine-tuning of our pre-trained model

def tokenize(data):
    inputs = tokenizer(data["dialogue"], padding = "max_length", max_length = 512, truncation = True)
    targets = tokenizer(data["summary"], padding = "max_length", max_length = 150, truncation = True)

    inputs["labels"] = targets["input_ids"] # token id => add to input as labels
    return inputs

In [15]:
train_df = train_data.apply(tokenize, axis = 1).tolist()
val_df = val_data.apply(tokenize, axis = 1).tolist()

In [16]:
train_df[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [17]:
## input_ids = dialogue -> token_id


# 1 -> EOS  # 0 -> padding


# attention_mask -> it tells us the valid values in the input_ids; 1 -> valid

# labels --> targets(summary)

print(len(train_df[0]["input_ids"]))
print(len(train_df[0]["labels"]))
print(type(train_df))
print(type(val_df))

512
150
<class 'list'>
<class 'list'>


# Fine Tuning our Pre-trained Model

In [18]:
# here we are summarizing our text. So, it's a NLP/generation task.
# now this generation is dependent on the input. So, it is conditional.

model = T5ForConditionalGeneration.from_pretrained("t5-small")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [19]:
# Fine-Tune

import torch

if torch.backends.mps.is_available():
    device = torch.device("nps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Device is {device}")
model.to(device)

Device is cuda


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [20]:
# Training Arguments

training_args = TrainingArguments(
    output_dir = "./results",

    num_train_epochs = 6,
    weight_decay = 0.01,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,

    eval_strategy = "epoch",     # want to validate our model after every epoch
    save_strategy = "epoch",     # want to save our model after every epoch

    warmup_steps = 500
    # in this our learning rate increase from 0 to its default value in 500 steps.
)

In [21]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_df,
    eval_dataset = val_df
)

In [22]:
# Train the model

trainer.train()

Epoch,Training Loss,Validation Loss
1,3.595242,0.380791
2,0.397143,0.359807
3,0.374322,0.354439
4,0.361865,0.350460
5,0.355648,0.349347
6,0.350701,0.349433


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3000, training_loss=0.9058200988769531, metrics={'train_runtime': 1171.6814, 'train_samples_per_second': 20.483, 'train_steps_per_second': 2.56, 'total_flos': 3248203235328000.0, 'train_loss': 0.9058200988769531, 'epoch': 6.0})

In [23]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./saved_summary_model/tokenizer_config.json',
 './saved_summary_model/tokenizer.json')

In [24]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

# Testing the core logic for Summarization

In [25]:
def summarize_dialogue(dialogue):

  # Data Preprocessing

  dialogue = clean_data(dialogue)

  # generate tokens of the dialogue
  inputs = tokenizer(
      dialogue,
      padding = "max_length",
      max_length = 512,
      truncation = True,
      return_tensors = "pt"
  ).to(device)

  # generate summary -> which will be in tokens
  model.to(device)
  targets = model.generate(
      input_ids = inputs["input_ids"],
      attention_mask = inputs["attention_mask"],
      max_length = 150,
      num_beams = 4,   # transformer will generate 4 diff. o/p and give us the best one among those 4 o/p.
      early_stopping = True
  )



  # tokens id convertion into text => decoding

  summary = tokenizer.decode(targets[0], skip_special_tokens = True)

  return summary


In [26]:
test_dialogue = """Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.
"""


summary = summarize_dialogue(test_dialogue)

print(f"Summary is :\n {summary}")

Summary is :
 ai adoption has significantly increased over the past few years. experts highlight the importance of responsible ai development, including data privacy, security and long-term societal impact.
